# Lab 2 - Redes Neurais

Classificador: **MLP (MultiLayer Perceptron)**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import os
from datetime import datetime

## Carregamento dos Dados Processados

In [ ]:
data_dir = './dados_processados'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').values.ravel()
X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').values.ravel()

mod_time = datetime.fromtimestamp(os.path.getmtime(f'{data_dir}/X_train.parquet'))
print(f'Dados carregados (gerados em {mod_time.strftime("%Y-%m-%d %H:%M")})')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}  |  y_test: {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')
print(f'  Churn rate treino: {y_train.mean():.3f}  |  teste: {y_test.mean():.3f}')

## Configuração do MLflow e Função de Avaliação

O MLflow registra automaticamente parâmetros, métricas e artefatos de cada modelo.
Todos os notebooks usam o mesmo experimento (`Lab2-Churn`), então os runs aparecem juntos na UI.

**Para visualizar os resultados após rodar os notebooks:**
```bash
cd notebooks
mlflow ui
# Abrir http://127.0.0.1:5000 no navegador
```

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

mlflow.set_experiment("Lab2-Churn")


def avaliar_modelo(nome, y_true, y_pred):
    """Avalia um modelo, printa o relatório e retorna dict com métricas."""
    metricas = {
        'Modelo': nome,
        'Acuracia': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    print(f'\n=== {nome} ===')
    print(classification_report(y_true, y_pred, target_names=['Não cancelou', 'Cancelou']))
    print(f'Kappa: {metricas["Kappa"]:.4f}')
    return metricas


def logar_mlflow(metricas, params, artefatos=None):
    """Registra métricas, parâmetros e artefatos no run MLflow ativo."""
    for k, v in params.items():
        mlflow.log_param(k, v)
    for k, v in metricas.items():
        if k != 'Modelo':
            mlflow.log_metric(k.lower().replace('-', '_'), v)
    if artefatos:
        for caminho in artefatos:
            mlflow.log_artifact(caminho)

## Normalização

MLP é sensível à escala das features — aplicamos `StandardScaler`.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Dados normalizados — média ~0, std ~1')
print(f'  X_train_scaled: {X_train_scaled.shape}')
print(f'  X_test_scaled:  {X_test_scaled.shape}')

## Exemplo de Uso — Padrão MLflow

O `with mlflow.start_run(...)` delimita um experimento. Tudo dentro do bloco
(parâmetros, métricas, artefatos, modelo) fica associado a esse run.

In [ ]:
# ─── Exemplo: MLP ───────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier

with mlflow.start_run(run_name="MLP"):
    # 1. Parâmetros
    hidden = (100, 50)
    lr = 0.001
    params = {
        'modelo': 'MLP',
        'hidden_layer_sizes': str(hidden),
        'learning_rate_init': lr,
        'max_iter': 500,
        'scaler': 'StandardScaler'
    }

    # 2. Treinar
    model = MLPClassifier(
        hidden_layer_sizes=hidden,
        learning_rate_init=lr,
        max_iter=500,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)

    # 3. Predizer e avaliar
    y_pred = model.predict(X_test_scaled)
    metricas = avaliar_modelo('MLP', y_test, y_pred)

    # 4. Logar no MLflow (incluindo curva de loss como artefato)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(model.loss_curve_)
    ax.set_title('Loss por Época — MLP')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    plt.tight_layout()
    loss_path = '../relatorio/imagens/2b_rn_loss_por_epoca.png'
    plt.savefig(loss_path, dpi=150, bbox_inches='tight')
    plt.show()

    logar_mlflow(metricas, params, artefatos=[loss_path])